# Simple RAG Workflow with LlamaIndex + Ollama

A step-by-step, fully local **Retrieval-Augmented Generation (RAG)** notebook. No API keys, no cloud calls, no per-token cost.

This notebook walks through the complete pipeline:

```
data/ documents  ->  chunks  ->  embeddings (nomic-embed-text)
                 ->  vector index (persisted to storage/)
                 ->  retrieve top-k chunks  ->  LLM (gemma3:4b)  ->  answer
```

**What you'll do here**

1. Configure LlamaIndex to use local Ollama models
2. Load documents from the `data/` folder
3. Chunk, embed, and index them into a vector store
4. Persist the index to disk so it can be reused
5. Ask questions and inspect the grounded answers + their sources

---

## Prerequisites

- Python 3.9+
- [Ollama](https://ollama.com/download) installed and running
- The two models this notebook uses:

```bash
ollama pull gemma3:4b
ollama pull nomic-embed-text
```


## Step 0 — Install dependencies

Run this once to install the LlamaIndex packages and their Ollama integrations. If they're already installed, this is a no-op.

In [1]:
# Uncomment to install (or run: pip install -r requirements.txt)
# %pip install llama-index llama-index-llms-ollama llama-index-embeddings-ollama

## Step 1 — Configuration

These are the knobs for the whole pipeline: which models to use, where the documents and index live, and how many chunks to retrieve per question.

In [2]:
OLLAMA_BASE_URL = "http://localhost:11434"  # Ollama's default endpoint
LLM_MODEL = "gemma3:4b"                      # chat model used to generate answers
EMBED_MODEL = "nomic-embed-text"            # embedding model used for retrieval

DATA_DIR = "data"          # folder of source documents
STORAGE_DIR = "storage"    # where the built index is persisted
TOP_K = 3                   # number of chunks retrieved per question
REQUEST_TIMEOUT = 120.0     # seconds to wait for the LLM

print(f"LLM: {LLM_MODEL} | Embeddings: {EMBED_MODEL} | top_k: {TOP_K}")

LLM: gemma3:4b | Embeddings: nomic-embed-text | top_k: 3


## Step 2 — Point LlamaIndex at the local Ollama models

`Settings` is LlamaIndex's global configuration. By setting `Settings.llm` and `Settings.embed_model` here, every index and query engine we create later will automatically use these local models.

- **`Ollama`** — the chat model that writes the final answer.
- **`OllamaEmbedding`** — turns text chunks (and your questions) into vectors so similar text can be found.

In [3]:
from llama_index.core import Settings
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

Settings.llm = Ollama(
    model=LLM_MODEL,
    base_url=OLLAMA_BASE_URL,
    request_timeout=REQUEST_TIMEOUT,
)
Settings.embed_model = OllamaEmbedding(
    model_name=EMBED_MODEL,
    base_url=OLLAMA_BASE_URL,
)

print("Models configured.")

Models configured.


## Step 3 — Load the documents

`SimpleDirectoryReader` reads every supported file in `data/` (`.md`, `.txt`, `.pdf`, and more) and returns a list of `Document` objects. Each document keeps metadata such as the source file name, which we'll surface later as a citation.

In [4]:
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader(DATA_DIR).load_data()
print(f"Loaded {len(documents)} document(s) from '{DATA_DIR}/'.\n")

# Preview the first document
preview = documents[0]
print("Source file:", preview.metadata.get("file_name"))
print("Char count :", len(preview.text))
print("\n--- first 300 chars ---")
print(preview.text[:300])

Loaded 1 document(s) from 'data/'.

Source file: llamaindex_overview.md
Char count : 2625

--- first 300 chars ---
# LlamaIndex and Retrieval-Augmented Generation (RAG)

## What is Retrieval-Augmented Generation?

Retrieval-Augmented Generation (RAG) is a technique that improves the answers of a
large language model (LLM) by giving it relevant information retrieved from an
external knowledge base at query 


## Step 4 — Build (or load) the vector index

`VectorStoreIndex.from_documents(...)` does the heavy lifting:

1. **Chunks** each document into smaller nodes.
2. **Embeds** every node with `nomic-embed-text`.
3. **Stores** the vectors in an in-memory vector store.

We then **persist** the index to `storage/` so future runs can skip the embedding step. The cell below loads the saved index if one exists, and only rebuilds when it doesn't (set `REBUILD = True` to force a fresh build after changing your documents).

In [5]:
import os
from llama_index.core import (
    VectorStoreIndex,
    StorageContext,
    load_index_from_storage,
)

REBUILD = False  # set True to re-index after editing documents in data/

index_exists = os.path.isdir(STORAGE_DIR) and os.listdir(STORAGE_DIR)

if REBUILD or not index_exists:
    print("Building vector index (embedding chunks)...")
    index = VectorStoreIndex.from_documents(documents, show_progress=True)
    index.storage_context.persist(persist_dir=STORAGE_DIR)
    print(f"Index built and persisted to '{STORAGE_DIR}/'.")
else:
    print(f"Loading existing index from '{STORAGE_DIR}/'...")
    storage_context = StorageContext.from_defaults(persist_dir=STORAGE_DIR)
    index = load_index_from_storage(storage_context)
    print("Index loaded.")

Loading existing index from 'storage/'...
Index loaded.


## Step 5 — Create a query engine

The query engine ties together **retrieval** and **generation**. For each question it:

1. Embeds the question and retrieves the `TOP_K` most similar chunks.
2. Stuffs those chunks into a prompt for `gemma3:4b`.
3. Returns an answer grounded in the retrieved context.

In [6]:
query_engine = index.as_query_engine(similarity_top_k=TOP_K)
print(f"Query engine ready (retrieving top {TOP_K} chunks per question).")

Query engine ready (retrieving top 3 chunks per question).


## Step 6 — Ask a question

Here's a small helper that prints the answer along with the **source chunks** that grounded it (with their relevance scores), so you can verify the model isn't making things up.

In [7]:
def ask(question: str):
    response = query_engine.query(question)

    print("=" * 70)
    print(f"Q: {question}")
    print("-" * 70)
    print(f"A: {response}")

    sources = getattr(response, "source_nodes", []) or []
    if sources:
        print("-" * 70)
        print("Sources:")
        for i, node in enumerate(sources, start=1):
            file_name = node.metadata.get("file_name", "unknown")
            snippet = node.get_content().strip().replace("\n", " ")[:140]
            print(f"  [{i}] ({node.score:.3f}) {file_name}: {snippet}...")
    print("=" * 70)
    return response


_ = ask("What is Retrieval-Augmented Generation?")

Q: What is Retrieval-Augmented Generation?
----------------------------------------------------------------------
A: Retrieval-Augmented Generation (RAG) is a technique that improves the answers of a large language model (LLM) by giving it relevant information retrieved from an external knowledge base at query time. Instead of relying only on what the model memorized during training, a RAG system first retrieves passages that are relevant to the user's question and then asks the LLM to generate an answer grounded in those passages. This reduces hallucinations and lets the model answer questions about private or recent data it was never trained on.
----------------------------------------------------------------------
Sources:
  [1] (0.693) llamaindex_overview.md: # LlamaIndex and Retrieval-Augmented Generation (RAG)
 
 ## What is Retrieval-Augmented Generation?
 
 Retrieval-Augmented Generation (RAG) ...


## Step 7 — Try more questions

Ask anything grounded in the documents under `data/`. Edit the questions below or add your own.

In [8]:
_ = ask("Which two Ollama models does this project use and what is each for?")
print()
_ = ask("What are the five stages of a RAG pipeline?")

Q: Which two Ollama models does this project use and what is each for?
----------------------------------------------------------------------
A: The project uses `gemma3:4b` for generation and `nomic-embed-text` to create embeddings.
----------------------------------------------------------------------
Sources:
  [1] (0.654) llamaindex_overview.md: # LlamaIndex and Retrieval-Augmented Generation (RAG)
 
 ## What is Retrieval-Augmented Generation?
 
 Retrieval-Augmented Generation (RAG) ...



Q: What are the five stages of a RAG pipeline?
----------------------------------------------------------------------
A: A typical RAG pipeline has five stages: Loading, Chunking, Embedding, Indexing & Retrieval, and Generation.
----------------------------------------------------------------------
Sources:
  [1] (0.619) llamaindex_overview.md: # LlamaIndex and Retrieval-Augmented Generation (RAG)
 
 ## What is Retrieval-Augmented Generation?
 
 Retrieval-Augmented Generation (RAG) ...


## Next steps

- **Use your own data:** drop `.md` / `.txt` / `.pdf` files into `data/`, set `REBUILD = True` in Step 4, and re-run.
- **Tune retrieval:** change `TOP_K` in Step 1 to retrieve more or fewer chunks.
- **Swap models:** point `LLM_MODEL` / `EMBED_MODEL` at any models you've pulled with `ollama pull`.
- **Run as a script:** `rag.py` provides the same pipeline as a CLI (`python rag.py "your question"` or `python rag.py` for an interactive loop).